# 05-04. Tracer Experiments in PSLNMD  
# PSLNMD で保存トレーサー実験を行う

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

## 今日の目的 / Goals

05-03 では、PSLNMD の輸送行列を作りました。  
In 05-03, we built the transport matrix for PSLNMD.

この Notebook では、その輸送行列に **トレーサー** を流します。  
In this notebook, we put **tracers** into that transport matrix.

ここでは、まだ DIC や O2 は扱いません。  
Here, we do not yet treat DIC or O2.

まずは、保存トレーサーを使って、水塊の経路を見えるようにします。  
First, we use conserved tracers to visualize water-mass pathways.

> **トレーサーを使って、水塊がどこから来て、どこへ行くかを見えるようにする。**  
> **Use tracers to make water-mass pathways visible.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: なぜトレーサー実験をするのか / Why do tracer experiments?

輸送行列を作っただけでは、流れの意味が直感的に分かりにくいです。  
After building a transport matrix, the meaning of the flow is not always intuitive.

そこで、染料を 1 つの box に入れて、どこへ広がるかを見ます。  
So we release dye into one box and see where it spreads.

これは、モデルの理解とデバッグの両方に役立ちます。  
This helps both model interpretation and debugging.

## 2. PSLNMD と輸送行列を準備する / Prepare PSLNMD and transport matrix

箱は、

```text
P, S, L, N, M, D
```

基本経路は、

```text
N → M → D → S → P → L → N
```

です。

さらに、表層と中層の交換として、

```text
S ↔ M
L ↔ M
```

も加えます。

In [ ]:
boxes = ["P", "S", "L", "N", "M", "D"]
surface_boxes = ["P", "S", "L", "N"]
interior_boxes = ["M", "D"]

VOCN_TOTAL = 1.292e18
VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())
volumes = np.array([VOLUME[b] for b in boxes])

Q_DEFAULT = 2.0e7
DT = 8.64e4
SEC_PER_YEAR = 365 * 86400

pathway = [("N", "M"), ("M", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]
idx = {b: i for i, b in enumerate(boxes)}

def build_flux_matrix(pathway, Q, boxes):
    F = np.zeros((len(boxes), len(boxes)))
    idx_local = {b: i for i, b in enumerate(boxes)}
    for source, target in pathway:
        i = idx_local[target]
        j = idx_local[source]
        F[i, j] += Q
        F[j, j] -= Q
    return F

def build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges=None):
    F = build_flux_matrix(pathway, Q, boxes)
    idx_local = {b: i for i, b in enumerate(boxes)}
    if exchanges is None:
        exchanges = []
    for a, b, q in exchanges:
        ia = idx_local[a]
        ib = idx_local[b]
        F[ib, ia] += q
        F[ia, ia] -= q
        F[ia, ib] += q
        F[ib, ib] -= q
    return F

base_exchanges = [("S", "M", 0.3e7), ("L", "M", 0.2e7)]
F = build_flux_matrix_with_exchange(pathway, Q_DEFAULT, boxes, base_exchanges)

pd.DataFrame(F, index=boxes, columns=boxes)

## 3. 保存チェック / Conservation check

保存トレーサーでは、総量が保存されるはずです。  
For a conserved tracer, the total amount should be conserved.

総量は、

```text
sum(concentration * volume)
```

です。

In [ ]:
def one_step_transport(c, F, dt=DT):
    return c + (F @ c) * dt / volumes

def total_amount(c):
    return np.sum(c * volumes)

def run_passive_tracer(source_box="N", years=1500, F=F):
    c = np.zeros(len(boxes))
    c[idx[source_box]] = 1.0
    hist = []
    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365, "total": total_amount(c)}
            for i, b in enumerate(boxes):
                row[b] = c[i]
            hist.append(row)
        c = one_step_transport(c, F)
    return pd.DataFrame(hist)

test = run_passive_tracer("N", years=500)

plt.figure()
plt.plot(test["year"], test["total"])
plt.xlabel("Time [yr]")
plt.ylabel("Total tracer amount")
plt.title("Conservation of passive tracer")
plt.grid(True)
plt.show()

print("Relative change:", (test["total"].iloc[-1] - test["total"].iloc[0]) / test["total"].iloc[0])

### Mini exercise 1 / ミニ演習 1

総トレーサー量は保存されていますか。  
Is total tracer amount conserved?

保存されない場合、輸送行列の符号、対角成分、体積で割る処理を確認します。  
If not, check matrix signs, diagonal terms, and volume division.

## 4. 染料実験 1: N に入れる / Dye experiment 1: release in N

N は北大西洋表層で、沈み込みの入口として考えます。  
N is the North Atlantic surface box and is treated as the entrance of sinking.

N に入れた染料は M と D に入りやすいはずです。  
Dye released in N should easily enter M and D.

In [ ]:
dye_N = run_passive_tracer("N", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(dye_N["year"], dye_N[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in N")
plt.legend()
plt.grid(True)
plt.show()

## 5. 染料実験 2: S に入れる / Dye experiment 2: release in S

S は南大洋表層です。  
S is the Subpolar surface box.

S に入れた染料は、表層経路 P, L, N を通ってから内部へ入るはずです。  
Dye released in S should follow the surface pathway P, L, N before entering the interior.

In [ ]:
dye_S = run_passive_tracer("S", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(dye_S["year"], dye_S[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in S")
plt.legend()
plt.grid(True)
plt.show()

## 6. 染料実験 3: P に入れる / Dye experiment 3: release in P

P は極域表層です。  
P is the Polar surface box.

P に入れた染料は、L と N を経由して内部へ入るはずです。  
Dye released in P should enter the interior through L and N.

In [ ]:
dye_P = run_passive_tracer("P", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(dye_P["year"], dye_P[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in P")
plt.legend()
plt.grid(True)
plt.show()

## 7. Source box による違い / Difference among source boxes

N, S, P のどこに染料を入れるかで、D への到達の仕方が変わります。  
The pathway to D differs depending on whether dye is released in N, S, or P.

In [ ]:
plt.figure()
plt.plot(dye_N["year"], dye_N["D"], label="source N")
plt.plot(dye_S["year"], dye_S["D"], label="source S")
plt.plot(dye_P["year"], dye_P["D"], label="source P")
plt.xlabel("Time [yr]")
plt.ylabel("Dye in D")
plt.title("Effect of source box on arrival in D")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Source": ["N", "S", "P"],
    "D at 300 yr": [
        dye_N.loc[dye_N["year"] == 300, "D"].iloc[0],
        dye_S.loc[dye_S["year"] == 300, "D"].iloc[0],
        dye_P.loc[dye_P["year"] == 300, "D"].iloc[0],
    ],
    "D at 1000 yr": [
        dye_N.loc[dye_N["year"] == 1000, "D"].iloc[0],
        dye_S.loc[dye_S["year"] == 1000, "D"].iloc[0],
        dye_P.loc[dye_P["year"] == 1000, "D"].iloc[0],
    ],
    "D at 1500 yr": [dye_N["D"].iloc[-1], dye_S["D"].iloc[-1], dye_P["D"].iloc[-1]],
})

### Mini exercise 2 / ミニ演習 2

D に最も早く届くのは、どの source box の染料ですか。  
Which source-box dye reaches D fastest?

その理由を、輸送経路を使って説明してください。  
Explain why using the transport pathway.

## 8. 起源トレーサー / Origin tracer

次に、「どの表層 box 起源の水か」を見るトレーサーを作ります。  
Next, we make a tracer that indicates which surface box the water originates from.

例えば N-origin tracer は、

```text
N surface: restored to 1
other surface boxes: restored to 0
interior boxes: transported
```

とします。

In [ ]:
def one_step_origin(c, F, origin_box, restoring_strength=1.0):
    new = one_step_transport(c, F)
    for b in surface_boxes:
        i = idx[b]
        target = 1.0 if b == origin_box else 0.0
        new[i] = (1.0 - restoring_strength) * new[i] + restoring_strength * target
    return new

def run_origin_tracer(origin_box="N", years=3000, F=F):
    c = np.zeros(len(boxes))
    hist = []
    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365}
            for i, b in enumerate(boxes):
                row[b] = c[i]
            hist.append(row)
        c = one_step_origin(c, F, origin_box)
    return pd.DataFrame(hist)

origin_N = run_origin_tracer("N", years=3000, F=F)
origin_S = run_origin_tracer("S", years=3000, F=F)
origin_P = run_origin_tracer("P", years=3000, F=F)

plt.figure()
plt.plot(origin_N["year"], origin_N["D"], label="N-origin in D")
plt.plot(origin_S["year"], origin_S["D"], label="S-origin in D")
plt.plot(origin_P["year"], origin_P["D"], label="P-origin in D")
plt.xlabel("Time [yr]")
plt.ylabel("Origin tracer in D")
plt.title("Surface-origin tracers in D")
plt.legend()
plt.grid(True)
plt.show()

## 9. M と D の起源を比較する / Compare origins in M and D

M と D で、どの起源トレーサーが大きいかを比較します。  
We compare which origin tracer is larger in M and D.

In [ ]:
origin_summary = pd.DataFrame({
    "Box": ["M", "D"],
    "N-origin": [origin_N["M"].iloc[-1], origin_N["D"].iloc[-1]],
    "S-origin": [origin_S["M"].iloc[-1], origin_S["D"].iloc[-1]],
    "P-origin": [origin_P["M"].iloc[-1], origin_P["D"].iloc[-1]],
})
origin_summary

In [ ]:
x = np.arange(len(origin_summary["Box"]))
width = 0.25

plt.figure()
plt.bar(x - width, origin_summary["N-origin"], width, label="N-origin")
plt.bar(x, origin_summary["S-origin"], width, label="S-origin")
plt.bar(x + width, origin_summary["P-origin"], width, label="P-origin")
plt.xticks(x, origin_summary["Box"])
plt.ylabel("Origin tracer")
plt.title("Origin tracers in M and D")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

### Mini exercise 3 / ミニ演習 3

M と D では、どの起源トレーサーが大きいですか。  
Which origin tracer is largest in M and D?

それは輸送経路と整合的ですか。  
Is it consistent with the transport pathway?

## 10. Source-sink tracer / Source-sink トレーサー

継続的な source を持つトレーサーを考えます。  
Now we consider a tracer with a continuous source.

例えば、P box に毎ステップ少しずつトレーサーを加えます。  
For example, we add a small tracer source to P at every time step.

In [ ]:
def one_step_source_sink(c, F, source_box="P", source_rate=1.0e-10, decay_rate=0.0):
    new = one_step_transport(c, F)
    new[idx[source_box]] += source_rate * DT
    if decay_rate > 0.0:
        new *= np.exp(-decay_rate * DT)
    return new

def run_source_sink(source_box="P", years=1500, source_rate=1.0e-10, decay_rate=0.0, F=F):
    c = np.zeros(len(boxes))
    hist = []
    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365, "total": total_amount(c)}
            for i, b in enumerate(boxes):
                row[b] = c[i]
            hist.append(row)
        c = one_step_source_sink(c, F, source_box, source_rate, decay_rate)
    return pd.DataFrame(hist)

source_P = run_source_sink("P", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(source_P["year"], source_P[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Source tracer")
plt.title("Continuous source in P")
plt.legend()
plt.grid(True)
plt.show()

## 11. Decay を入れる / Add decay

放射性同位体のようなトレーサーでは、時間とともに減衰します。  
For tracers like radioisotopes, concentration decreases with time.

この考え方は、後で \( \Delta^{14}\mathrm{C} \) を学ぶときにつながります。  
This connects to \( \Delta^{14}\mathrm{C} \) later.

In [ ]:
tau_years = 1000
decay_rate = 1.0 / (tau_years * SEC_PER_YEAR)

source_no_decay = run_source_sink("P", years=1500, decay_rate=0.0, F=F)
source_decay = run_source_sink("P", years=1500, decay_rate=decay_rate, F=F)

plt.figure()
plt.plot(source_no_decay["year"], source_no_decay["D"], label="D no decay")
plt.plot(source_decay["year"], source_decay["D"], label="D with decay")
plt.xlabel("Time [yr]")
plt.ylabel("Tracer in D")
plt.title("Effect of decay on tracer reaching D")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(source_no_decay["year"], source_no_decay["total"], label="total no decay")
plt.plot(source_decay["year"], source_decay["total"], label="total with decay")
plt.xlabel("Time [yr]")
plt.ylabel("Total tracer amount")
plt.title("Decay reduces total tracer amount")
plt.legend()
plt.grid(True)
plt.show()

### Mini exercise 4 / ミニ演習 4

Decay を入れると、D に届くトレーサー量はどう変わりましたか。  
How did the tracer amount reaching D change when decay was added?

なぜ、古い水ほど decay の影響を受けやすいのでしょうか。  
Why is older water more affected by decay?

## 12. 輸送感度実験 / Transport sensitivity experiment

最後に、流量 Q を変えます。  
Finally, we change transport strength Q.

Q が大きいほど、水は速く動きます。  
Larger Q means faster water movement.

In [ ]:
Q_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary_Q = []

plt.figure()
for q in Q_list:
    Fq = build_flux_matrix_with_exchange(pathway, q, boxes, exchanges=base_exchanges)
    h = run_passive_tracer("N", years=1500, F=Fq)
    plt.plot(h["year"], h["D"], label=f"Q={q:.1e}")
    summary_Q.append({
        "Q": q,
        "D at 300 yr": h.loc[h["year"] == 300, "D"].iloc[0],
        "D at 1000 yr": h.loc[h["year"] == 1000, "D"].iloc[0],
        "D at 1500 yr": h["D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Dye in D")
plt.title("Transport strength and arrival in D")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_Q)

## 13. 05-04 のまとめ / Summary of 05-04

この Notebook で学んだことは次です。  
What we learned:

1. 染料トレーサーは、水塊の経路を可視化する。  
   Dye tracers visualize water-mass pathways.

2. N に入れた染料は、M と D に入りやすい。  
   Dye released in N easily enters M and D.

3. S や P に入れた染料は、表層経路を通ってから内部へ入る。  
   Dye released in S or P enters the interior after following surface pathways.

4. 起源トレーサーは、M や D がどの表層 box とつながっているかを見るために使える。  
   Origin tracers show which surface boxes are connected to M and D.

5. Decay を持つトレーサーは、水塊年齢や放射性同位体の理解につながる。  
   Tracers with decay connect to water-mass age and radiocarbon interpretation.

## Key message

> **トレーサー実験は、輸送行列を「流れの物語」として理解するための道具である。**  
> **Tracer experiments are tools for understanding a transport matrix as a story of water movement.**

## 14. 課題 / Exercises

### 課題 1 / Exercise 1

N に入れた染料が M と D に入りやすい理由を説明せよ。  
Explain why dye released in N easily enters M and D.

### 課題 2 / Exercise 2

S に入れた染料と P に入れた染料では、D への届き方がどう違うか。  
How does the arrival at D differ between dye released in S and dye released in P?

### 課題 3 / Exercise 3

Origin tracer は何を見るためのものか。  
What is an origin tracer used for?

### 課題 4 / Exercise 4

Source-sink tracer と dye tracer の違いを説明せよ。  
Explain the difference between a source-sink tracer and a dye tracer.

### 課題 5 / Exercise 5

Decay を持つトレーサーが、なぜ水塊年齢の理解につながるのか。  
Why does a tracer with decay help us understand water-mass age?

## 想定解答 / Expected answers

### 課題 1

N は北大西洋表層であり、沈み込みの入口として設定されている。そのため N に入った染料は表層に留まらず、M を通って D へ輸送されやすい。  
N is the North Atlantic surface box and is set as the entrance of sinking. Therefore, dye released in N does not remain at the surface, but is transported through M into D.

### 課題 2

S に入れた染料は S → P → L → N という表層経路を通ってから内部へ入る。一方、P に入れた染料は P → L → N を通って内部へ入る。したがって、source box によって D への到達時間が変わる。  
Dye released in S follows the surface pathway S → P → L → N before entering the interior. Dye released in P follows P → L → N before entering the interior. Thus, arrival time at D depends on the source box.

### 課題 3

Origin tracer は、内部の M や D がどの表層 box の影響を強く受けているかを見るためのトレーサーである。  
An origin tracer is used to diagnose which surface box most strongly influences interior boxes such as M and D.

### 課題 4

Dye tracer は最初に一度だけ入れるトレーサーである。Source-sink tracer は、ある box に継続的な供給や消費を持つトレーサーである。  
A dye tracer is released once at the beginning. A source-sink tracer has continuous supply or removal in one or more boxes.

### 課題 5

Decay を持つトレーサーは、時間が経つほど濃度が低下する。そのため、表層から長く隔離された古い水では信号が弱くなり、水塊年齢の情報を持ちうる。  
A tracer with decay decreases over time. Therefore, in old water isolated from the surface for a long time, the signal becomes weaker and can contain information about water-mass age.

## 15. 次へ / Next step

次の Notebook では、ideal age を導入します。  
In the next notebook, we introduce ideal age.

```text
05-05 Ideal Age in PSLNMD
```